In [1]:
#TODO: Import your dependencies.
#For instance, below are some dependencies you might need if you are using Pytorch
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
# from torchvision.models import resnet50, ResNet50_Weights

import argparse 
import logging
import sys
import os
import smdebug

# import smdebug.Pytorch as smd

from smdebug import modes
from smdebug.profiler.utils import str2bool
from smdebug.pytorch import get_hook


[2024-12-26 20:41:30.393 default:1789 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2024-12-26 20:41:31.098 default:1789 INFO profiler_config_parser.py:111] Unable to find config at /opt/ml/input/config/profilerconfig.json. Profiler is disabled.


In [5]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [9]:
model.fc.in_features

512

In [10]:
model = models.resnet18(pretrained=False)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [11]:
with open(os.path.join(model_dir, "model.pth"), "rb") as f:
    model.load_state_dict(torch.load(f), map_location= device )

512

In [9]:
def get_pretained_model_ResNet50():
    weights = models.ResNet50_Weights()
    model = models.resnet50(weights=weights)

    return model

In [5]:
def test(model, test_dataloader, criterion, device=torch.device("cpu")):
    '''
    TODO: Complete this function that can take a model and a 
          testing data loader and will get the test accuray/loss of the model
          Remember to include any debugging/profiling hooks that you might need
    '''

    logger.info('HPO: model test starting')
    hook.set_mode(smd.modes.EVAL) # assign the debugger hook

    model.eval()
    with torch.no_grad():
        test_running_loss = 0
        accuracy_running = 0

        for data, target in test_dataloader:
            data   = data.to_device(device)
            target = data.to_device(device)

            pred = model(data)
            loss = criterion(pred, target)
            test_running_loss += loss.item()

            prob = torch.exp(pred)
            top_label, top_class = prob.topk(1, dim=1)

            correct = top_class == target.view(*top_class.shape)
            accuracy_running += torch.mean(correct.type(torch.FloatTensor)).item()

        test_loss = test_running_loss / len(test_dataloader)
        accuracy  = accuracy_running  / len(test_dataloader)

        logger.info("HPO: Test loss : {:.3f},\
            accuracy : {:.3f} ".format(test_loss,
                                        accuracy
                                        ))


        logger.info("HPO: model testing completed")


NameError: name 'torch' is not defined

In [1]:
def train(model, train_dataloader, valid_dataloader, criterion, optimizer, epochs=2, device=torch.device("cpu")):
    '''
    TODO: Complete this function that can take a model and
          data loaders for training and will get train the model
          Remember to include any debugging/profiling hooks that you might need
    '''

    hook = get_hook(create_if_not_exists=True)

    train_loss = []
    valid_loss = []
    accuracy   = []
    prev_accuracy = 0

    device = torch.device("cuda" if (torch.cuda.is_available() & gpu) else "cpu")
        
    logger.info('HPO: Training started on device {}'.format(device))

    if hook:
        hook.register_loss(optimizer)
  
    for epoch in range(epochs):

        train_running_loss = 0
        valid_running_loss = 0
        accuracy_running   = 0
        
        # hook.set_mode(smd.modes.TRAIN) # set debugging hook

        if hook:
            hook.set_mode(smd.modes.TRAIN) # assign the debugger hook

        model.train()
        for data, target in train_dataloader:
            data   = data.to_device(device)
            target = target.to_device(device)

            optimizer.zero_grad()

            pred = model(data)
            loss = criterion(pred, target)
            train_running_loss += loss.item()
        
            loss.backward()
            optimizer.step()

            # pred = pred.argmax(dim=1, keepdim=True)

            # correct += pred.eq(target.view_as(pred)).sum().item()

            # total_loss = running_loss / len(train_loader.dataset)
            # accuracy = correct / len(train_loader.dataset)

            # print("epoch : {}, total loss : {}, accuracy :{}%".format(e, total_loss, accuracy))
            
        if hook:
            hook.set_mode(smd.modes.EVAL) # assign the debugger hook

        model.eval()
        with torch.no_grad():
            valid_running_loss = 0

            for data, target in valid_dataloader:
                data   = data.to_device(device)
                target = data.to_device(device)

                pred = model(data)
                loss = criterion(pred, target)
                valid_running_loss += loss.item()

                prob = torch.exp(pred)
                top_label, top_class = prob.topk(1, dim=1)

                correct = top_class == target.view(*top_class.shape)
                accuracy_running += torch.mean(correct.type(torch.FloatTensor)).item()

        # training and evaluation loop completed

        train_running_loss = train_running_loss / len(train_dataloader)
        valid_running_loss = valid_running_loss / len(valid_dataloader)
        accuracy_running   = accuracy_running   / len(valid_dataloader)


        train_loss.append(train_running_loss)
        valid_loss.append(valid_running_loss)
        accuracy.append(accuracy_running)

    # pass
        logger.info("HPO: epoch {} of {},\
            training loss : {:.3f},\
            validation loss : {:.3f},\
            accuracy : {:.3f} ".format(epoch+1, epochs,
                                            train_running_loss,
                                            valid_running_loss,
                                            accuracy_running
                                            ))


    # return of losses and accuracy for graph representation, if desired
    return train_loss, valid_loss, accuracy

NameError: name 'torch' is not defined

In [7]:
def create_data_loaders(data_train, data_valid, data_test, batch_size):
    '''
    This is an optional function that you may or may not need to implement
    depending on whether you need to use data loaders or not
    '''

    logger.info("HPO: creating data loaders")
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    training_transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)])

    testing_transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)])


    trainset = ImageFolder(data_train, transform=training_transform)
    validset = ImageFolder(data_valid, transform=testing_transform)
    testset  = ImageFolder(data_test , transform=testing_transform)

    logger.info("HPO: Batch Size {}".format( batch_size))
    
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(validset, batch_size=batch_size, shuffle=True)
    test_loader  = torch.utils.data.DataLoader(testset , batch_size=batch_size, shuffle=True)

    logger.info('HPO: Data loaders created')
    return train_loader, valid_loader, test_loader


    # pass


In [42]:
def net(num_classes):
    '''
    TODO: Complete this function that initializes your model
          Remember to use a pretrained model
    '''
# def get_pretained_model_ResNet50():
#     model = models.resnet50(weights=weights)


    # model = torchvision.models.detection.ResNet50_Weights()
    # model = get_pretained_model_ResNet50()
    # weights = models.ResNet50_Weights()
    # model = models.resnet50(weights=weights)

    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    for params in model.parameters():
        params.requires_grad = False

    num_features = model.fc.in_features

    model.fc = nn.Sequential(
                   nn.ReLU(nn.Linear(num_features, 1024)),
                   nn.ReLU(nn.Linear(1024        , 512)),
                   nn.ReLU(nn.Linear(512         , 256)),
                   nn.ReLU(nn.Linear(256         , num_classes)),
                   nn.Softmax(dim=1)
                   )

    for param in model.fc.parameters():  # Assuming you're fine-tuning the fully connected layer
        param.requires_grad = True

    # logger.info("HPO: Model training completed")
    return model


In [43]:
model=net(133)


In [ ]:

# net = models.__dict__[opt.model](pretrained=True)

device = torch.device("gpu" if (torch.cuda.is_available() & agrs.gpu) else "cpu")

# if (args.gpu):
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")

model.to(device)

logger.info(f"HPO: Running on Device {device}")

logger.info(f'HPO: Hyperparameters are LR: {args.lr}, Batch Size: {args.batch_size}')
logger.info(f'HPO: Data Paths: {args.data_path}')

# train_data = args.data_path + "/train/"
# test_data  = args.data_path + "/test/"
# valid_data = args.data_path + "/valid/"

logger.info('HPO: create the data loaders')
train_loader, valid_loader, test_loader=create_data_loaders(data_train,  data_valid, data_test, args.batch_size)


'''
TODO: Create your loss and optimizer
'''
# loss_criterion = None
# optimizer = None

# loss_criterion = nn.CrossEntropyLoss() # using cross Entropy loss function
loss_criterion = nn.NLLLoss() # using negative log likelihood loss 

optimizer = optim.Adam(model.fc.parameters(), lr=args.lr) #using adam optimizer

hook.register_loss(loss_criterion)

'''
TODO: Call the train function to start training your model
Remember that you will need to set up a way to get training data from S3
'''
logger.info('HPO: train the model')
model=train(model, train_loader, valid_loader, loss_criterion, optimizer, args.epoch, device)


'''
TODO: Test the model to see its accuracy
'''
logger.info("HPO: Test the Model")
test(model, test_loader, criterion)

'''
TODO: Save the trained model
'''
logger.info("HPO: Saving Model")
torch.save(model.state_dict(), os.path.join(args.model_dir, "model.pth")) # save the trained model to S3




In [24]:
    model = models.resnet50()


In [27]:
model.parameters

<bound method Module.parameters of ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 2

In [28]:

for params in model.parameters():
    params.requires_grad = False



In [29]:
model.fc.in_features

2048

In [30]:

num_features = model.fc.in_features


In [31]:

model.fc = nn.Sequential(
                nn.ReLU(nn.Linear(num_features, 1024)),
                nn.ReLU(nn.Linear(1024        , 512)),
                nn.ReLU(nn.Linear(512         , 256)),
                nn.ReLU(nn.Linear(256         , 133)),
                nn.Softmax(dim=1)
                )


for param in model.fc.parameters():  # Assuming you're fine-tuning the fully connected layer
    param.requires_grad = True

In [32]:
model.fc.parameters

<bound method Module.parameters of Sequential(
  (0): ReLU(
    inplace=True
    (inplace): Linear(in_features=2048, out_features=1024, bias=True)
  )
  (1): ReLU(
    inplace=True
    (inplace): Linear(in_features=1024, out_features=512, bias=True)
  )
  (2): ReLU(
    inplace=True
    (inplace): Linear(in_features=512, out_features=256, bias=True)
  )
  (3): ReLU(
    inplace=True
    (inplace): Linear(in_features=256, out_features=133, bias=True)
  )
  (4): Softmax(dim=1)
)>

In [11]:
batch_size = 32

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

training_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)])

testing_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)])


trainset = torchvision.datasets.ImageFolder("./dogImages/train/", transform=training_transform)
validset = torchvision.datasets.ImageFolder('./dogImages/valid/', transform=testing_transform)
testset = torchvision.datasets.ImageFolder('./dogImages/test/', transform=testing_transform)


train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
valid_loader = torch.utils.data.DataLoader(validset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=True)



In [46]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [1]:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)


NameError: name 'models' is not defined

In [6]:
num_classes = 133
# model = models.resnet15(weights=models.ResNet15_Weights.DEFAULT)#  .resnet15(weights=models.ResNet15_Weights.DEFAULT)
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)


for params in model.parameters():
    params.requires_grad = False

num_features = model.fc.in_features

model.fc = nn.Sequential(
                nn.Linear(num_features, 1024),
                nn.ReLU(),
                nn.Linear(1024        , 512),
                nn.ReLU(),
                nn.Linear(512         , 256),
                nn.ReLU(),
                nn.Linear(256         , num_classes),
                nn.Softmax(dim=1)
                )

# for param in model.fc.parameters():  # Assuming you're fine-tuning the fully connected layer
#     param.requires_grad = True

In [7]:
loss_criterion = nn.NLLLoss() # using negative log likelihood loss 

optimizer = optim.Adam(model.fc.parameters(), lr=0.05) #using adam optimizer


# loss_criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.fc.parameters(), lr=0.05)

In [97]:
model = models.resnet34(pretrained=True) # using the pretrained resnet34 model with 34 layers

for param in model.parameters():
    param.requires_grad = False # freeze the model

nfeatures = model.fc.in_features
model.fc = nn.Sequential(
                nn.Linear(nfeatures, 512),
                nn.ReLU(),
                nn.Linear(512, 256), # adding own NN layers to the output of the pretrained model
                nn.ReLU(),
                nn.Linear(256, 133)) # output should be 133 as we have 133 classes of dog breeds

    

In [8]:
# def train(model, train_dataloader, valid_dataloader, criterion, optimizer, epochs=2, device=torch.device("cpu")):

    # hook = get_hook(create_if_not_exists=True)

train_loss = []
valid_loss = []
accuracy   = []
prev_accuracy = 0

epochs = 2
device = torch.device("cuda" if (torch.cuda.is_available()  ) else "cpu")
    

for epoch in range(epochs):

    train_running_loss = 0
    valid_running_loss = 0
    accuracy_running   = 0
    
    # hook.set_mode(smd.modes.TRAIN) # set debugging hook

    model.train()
    for data, target in train_loader:

        data   = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        pred = model(data)
        loss = loss_criterion(pred, target)

        print(f"loss = {loss}")

        print(f"target shape {target.shape}, pred shape {pred.shape}")


        loss.backward()
        optimizer.step()

        train_running_loss += loss.item()
        print("continuing loop")
        

        # pred = pred.argmax(dim=1, keepdim=True)

        # correct += pred.eq(target.view_as(pred)).sum().item()

        # total_loss = running_loss / len(train_loader.dataset)
        # accuracy = correct / len(train_loader.dataset)

        # print("epoch : {}, total loss : {}, accuracy :{}%".format(e, total_loss, accuracy))
        


loss = -0.007520065642893314
target shape torch.Size([32]), pred shape torch.Size([32, 133])
continuing loop


KeyboardInterrupt: 

In [74]:
print(pred.shape)

torch.Size([32, 2048])


In [36]:
for data, target in train_loader:
    print(data.requires_grad)
    print(target)
    break

False
tensor([ 78, 130,  44,  20, 110,   6,   4,  71,  53, 105, 104,  16,  70,  82,
         26,  42,  55,  62,  90, 115, 118,  69,   3,  33,  62,  15,  63,  14,
         92,   1,  36,  15])


In [12]:
    model.eval()
    with torch.no_grad():
        test_running_loss = 0
        accuracy_running = 0

        for data, target in test_loader:
            data   = data.to(device)
            target = target.to(device)

            pred = model(data)
            loss = criterion(pred, target)
            test_running_loss += loss.item()

            prob = torch.exp(pred)
            top_label, top_class = prob.topk(1, dim=1)

            correct = top_class == target.view(*top_class.shape)
            accuracy_running += torch.mean(correct.type(torch.FloatTensor)).item()

        test_loss = test_running_loss / len(test_loader)
        accuracy  = accuracy_running  / len(test_loader)

        logger.info("HPO: Test loss : {:.3f},\
            accuracy : {:.3f} ".format(test_loss,
                                        accuracy
                                        ))


        logger.info("HPO: model testing completed")


NameError: name 'criterion' is not defined